# Router Before the LLM: submission notebook

A cost-aware intent router for a personal finance assistant. This notebook is thin on purpose: it imports from `src/router/` and calls functions, so the same code runs here, in `scripts/` and in the demo.

**How to run:** Restart and Run All. With `FAST = True` (the default) it needs no GPU, no API key and no paid call. Numbers computed in fast mode are labelled as fast-mode numbers. The reported results are loaded from `results/`, where the scripts wrote them.

Sections follow the marking rubric: problem, data and preprocessing, models, evaluation, router and cost, limitations.

In [ ]:
# Cell 1: environment.
# On Colab: clone the repo and install the pinned packages. Locally: nothing to install.
import os
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # set once the GitHub repo exists (needed for the Colab clean-run check)

if IN_COLAB:
    import subprocess
    if not REPO_URL:
        raise RuntimeError("REPO_URL is not set yet, so the repo cannot be cloned on Colab.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "router-project"], check=True)
    os.chdir("router-project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    sys.path.insert(0, os.path.abspath("src"))  # make `import router` work without a restart

print("Python", sys.version.split()[0])
for pkg in ["torch", "transformers", "datasets", "sentence-transformers", "scikit-learn", "pandas"]:
    print(f"{pkg:>22} {version(pkg)}")

In [ ]:
# Cell 2: run mode and seeds.
FAST = True  # True: quick run, no GPU or API key needed. False: re-run the full five-seed scripts (S1 training measured at 30 min on an Apple M3 Pro GPU, 22 Nov 2026).
SEED = 42

import time

from router.seed import pick_device, set_seed

set_seed(SEED)
NOTEBOOK_START = time.perf_counter()
print("FAST =", FAST, "| SEED =", SEED, "| device:", pick_device())

## 1. Problem

A finance assistant sends every user query to an LLM, which costs money and time on each call. A small local model (the router) looks at each query first and either handles it directly, rejects it as out of scope, or escalates it to the LLM when unsure.

**Research question:** how many LLM calls can a small router safely remove, and at what error cost? The headline metric is coverage at matched risk: the largest share of queries the router can decide itself while its error rate stays at or below the LLM's.

## 2. Data and preprocessing

In-scope queries come from **Banking77** (Casanueva et al., 2020): 77 banking intents. Out-of-scope queries come from **CLINC150** (Larson et al., 2019). Both are loaded at a pinned Hub revision. Splits are created once by `scripts/make_splits.py` and saved to `data/processed/`, so every system sees the same data.

Pipeline (`src/router/preprocess.py`): unicode NFKC normalisation and whitespace cleanup, removal of empty rows and duplicates, a train/test leakage check, a stratified 10% validation split, token length checks and class balance. The full record is in `results/data_report.md`.

In [ ]:
import json

import pandas as pd

from router import data

splits = {name: data.load_split(name) for name in data.SPLIT_FILES}
pd.DataFrame({"rows": {name: len(df) for name, df in splits.items()}})

In [ ]:
# Rows removed at each step, per source (every removed row is listed in removed_rows.csv)
removed = pd.read_csv(data.PROCESSED_DIR / data.REMOVED_ROWS_FILE, keep_default_na=False)
removed.groupby(["source", "reason"]).size().unstack(fill_value=0)

In [ ]:
# Token lengths per tokeniser, measured on train. max_length is used by S1 (and later S2, S3).
with open(data.PROCESSED_DIR / "token_lengths.json") as f:
    lengths = json.load(f)
pd.DataFrame(
    {
        name: {
            "p50": v["single"]["p50"],
            "p99": v["single"]["p99"],
            "max": v["single"]["max"],
            "max_length": v["single"]["max_length"],
            "test truncated": f"{v['truncated_share']['test']:.2%}",
            "pair max_length (S3)": v["pair"]["max_length"],
        }
        for name, v in lengths.items()
    }
).T

In [ ]:
# Class balance: queries per intent in train. Uneven counts are why macro averages are primary.
import matplotlib.pyplot as plt

counts = splits["train"]["label"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(counts)), counts.values)
ax.set_xlabel("intent (sorted by count)")
ax.set_ylabel("train queries")
ax.set_title(f"Banking77 train: {counts.min()} to {counts.max()} queries per intent")
plt.show()

## 3. Models

Every system returns the same three things per query: a predicted intent, an in-scope confidence (the maximum softmax probability) and an out-of-scope score (1 minus that confidence). That shared interface (`src/router/systems/base.py`) is what makes the comparison fair.

- **S0, TF-IDF + logistic regression** (`s0_tfidf.py`): the cheapest possible baseline. It shows how much of the task keywords alone can solve. Its n-gram range and regularisation strength were chosen on validation macro-F1 (`results/s0_grid.json`).
- **S1, fine-tuned DistilRoBERTa** (`s1_encoder.py`): the standard approach for this topic. A small pretrained encoder with a softmax head, fine-tuned for 5 epochs with fixed, standard hyperparameters. DeBERTa-v3-small was the other candidate but could not be trained on the available hardware (`results/s1_selection.json`).

S2, S3 (M2) and S4 (M4) are added later.

In [ ]:
# FAST = False: re-run the same scripts that produced the reported results.
import subprocess

if not FAST:
    for command in [
        ["scripts/train.py", "--system", "s0"],
        ["scripts/train.py", "--system", "s1"],
        ["scripts/evaluate.py", "--split", "val"],
    ]:
        subprocess.run([sys.executable, *command], cwd=data.PROJECT_ROOT, check=True)
else:
    print("FAST mode: skipping the full five-seed re-run; reported results are loaded from results/.")

In [ ]:
# S0: train with the parameters chosen on validation, then predict the validation set.
from router.systems import s0_tfidf, s1_encoder
from router.systems.base import label_names_in_id_order

label_names = label_names_in_id_order()
train_df = splits["train"]
val_in_scope = splits["val"]  # E1 scores in-scope queries only

s0_cfg = data.load_config("s0_tfidf")
with open(data.RESULTS_DIR / "s0_grid.json") as f:
    s0_params = json.load(f)["chosen"]

start = time.perf_counter()
s0_model = s0_tfidf.train(train_df, s0_params, s0_cfg, seed=s0_cfg["seeds"][0])
s0_preds = s0_tfidf.predict(s0_model, val_in_scope["text"].tolist(), label_names)
print(f"S0 {s0_params}: trained and predicted in {time.perf_counter() - start:.1f}s")

In [ ]:
# S1: load the saved seed-0 checkpoint if this machine has it; otherwise train briefly (fast mode).
s1_cfg = data.load_config("s1_encoder")
checkpoint = s1_cfg["checkpoint"]
device = pick_device()
max_length = s1_encoder.max_length_for(checkpoint)
saved = data.PROJECT_ROOT / "checkpoints" / "s1" / s1_encoder.system_tag(checkpoint) / "seed0"

start = time.perf_counter()
if saved.exists():
    s1_model, s1_tokenizer = s1_encoder.load(saved, device)
    s1_source = "saved seed-0 checkpoint"
else:
    steps = s1_cfg["fast_steps"]
    s1_model, s1_tokenizer, _ = s1_encoder.train(train_df, s1_cfg, checkpoint, seed=0, device=device, label_names=label_names, max_steps=steps)
    s1_source = f"trained here for {steps} steps only (no checkpoint on this machine)"
s1_preds = s1_encoder.predict(s1_model, s1_tokenizer, val_in_scope["text"].tolist(), label_names, device, max_length)
print(f"S1 {checkpoint} ({s1_source}) on {device}: {time.perf_counter() - start:.1f}s")

## 4. Evaluation

E1 scores in-scope intent classification with accuracy, precision, recall and F1 (`src/router/metrics.py`, built on scikit-learn). Macro averages are primary because each of the 77 intents should count equally. Weighted averages are shown next to them.

**These are validation numbers.** The test split is not touched until every choice is frozen and the hypotheses are pre-registered (M5).

First, the numbers computed in this run (fast mode):

In [ ]:
from router.metrics import classification_metrics

METRIC_ORDER = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1"]
fast_rows = {
    "S0 (this run)": classification_metrics(val_in_scope["label"], s0_preds.pred_label, labels=label_names),
    f"S1 ({s1_source})": classification_metrics(val_in_scope["label"], s1_preds.pred_label, labels=label_names),
}
(pd.DataFrame(fast_rows).T[METRIC_ORDER] * 100).round(2)

**Reported results**: validation, mean ± standard deviation over five seeds, produced by `scripts/evaluate.py` (`results/e1_val.json`). S0 is deterministic, so its standard deviation is 0.

In [ ]:
with open(data.RESULTS_DIR / "e1_val.json") as f:
    e1 = json.load(f)

reported = {
    tag: {k: f"{100 * r['summary'][k]['mean']:.2f} ± {100 * r['summary'][k]['std']:.2f}" for k in METRIC_ORDER}
    for tag, r in e1["systems"].items()
}
pd.DataFrame(reported).T

In [ ]:
# Where S1 struggles: the 10 intents with the lowest F1 and the most frequent confusions (validation, all seeds).
s1_tag = s1_encoder.system_tag(checkpoint)
worst = pd.DataFrame(e1["systems"][s1_tag]["worst_intents"]).set_index("intent").round(3)
confused = pd.DataFrame(e1["systems"][s1_tag]["top_confused_pairs"])
display(worst)
display(confused)

## 5. Router and cost

Added in M4: router policy, risk-coverage curves, coverage at matched risk, LLM calls and cost saved.

## 6. Limitations

Written in M6, once the results are in.

In [ ]:
print(f"Notebook finished in {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} min (FAST = {FAST}).")